<a href="https://colab.research.google.com/github/vlasovdima/ai-trainer-portfolio/blob/main/Algorithms_(Python).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Вот готовый шаблон для первого кейса. Я оформил его по структуре, которую любят проверяющие: с четким разделением на задачу, идеальный ответ модели и аналитику от лица тренера. Можешь смело копировать этот текст в свой репозиторий.

***

###  Кейс №1: Оптимизация алгоритма поиска (SFT Dataset)

**Промпт пользователя:**
> «Напиши функцию на Python для поиска товаров в маркетплейсе по неточному совпадению названия. Используй расстояние Левенштейна, но оптимизируй алгоритм так, чтобы он не пересчитывал всю матрицу для длинных строк, если разница уже превышает заданный порог (допустим, 3 опечатки). Объясни асимптотическую сложность твоего решения.»

####  Эталонный ответ ассистента:

Для решения этой задачи в условиях маркетплейса стандартный алгоритм Левенштейна работает слишком медленно, так как вычисляет полную матрицу расстояний.

Чтобы избежать лишних вычислений, мы применим два уровня оптимизации:
1. **Предварительный фильтр по длине:** Если разница в длинах строк изначально больше порога (threshold), мы сразу возвращаем отказ.
2. **Алгоритм Укконена (Ukkonen's banded algorithm):** Мы не будем вычислять всю матрицу. Вместо этого мы будем считать значения только в узком "коридоре" (band) вокруг главной диагонали матрицы. Ширина этого коридора зависит от допустимого порога опечаток.


Вот реализация на Python:

In [1]:
def bounded_levenshtein(s1: str, s2: str, threshold: int = 3) -> int:
    """
    Вычисляет расстояние Левенштейна с отсечением по порогу (threshold).
    Возвращает точное расстояние, если оно <= threshold, иначе возвращает threshold + 1.
    """
    # 1. Быстрая проверка разницы длин
    if abs(len(s1) - len(s2)) > threshold:
        return threshold + 1

    # 2. Оптимизация памяти: s1 всегда должна быть более короткой строкой
    if len(s1) > len(s2):
        s1, s2 = s2, s1

    len1, len2 = len(s1), len(s2)

    # Храним только две строки матрицы (O(min(N, M)) памяти)
    prev_row = list(range(len1 + 1))
    curr_row = [0] * (len1 + 1)

    for i in range(1, len2 + 1):
        curr_row[0] = i

        # Вычисляем границы "коридора" вокруг диагонали
        start = max(1, i - threshold)
        end = min(len1, i + threshold)

        current_min = float('inf')

        for j in range(start, end + 1):
            cost = 0 if s1[j - 1] == s2[i - 1] else 1

            # Логика перехода с учетом краев коридора
            insertions = curr_row[j - 1] + 1 if j > start else float('inf')
            deletions = prev_row[j] + 1 if j <= end else float('inf')
            substitutions = prev_row[j - 1] + cost

            dist = min(insertions, deletions, substitutions)
            curr_row[j] = dist
            current_min = min(current_min, dist)

        # 3. Раннее отсечение: если минимум в строке превысил порог, останавливаемся
        if current_min > threshold:
            return threshold + 1

        prev_row = curr_row.copy()

    return prev_row[len1]

# Пример использования:
query = "смартфон"
catalog_item_1 = "смартфон"    # 0 опечаток
catalog_item_2 = "смартфно"    # 2 опечатки (перестановка = 2 операции)
catalog_item_3 = "умный телефон" # > 3 опечаток

print(bounded_levenshtein(query, catalog_item_2, threshold=3)) # Выведет 2
print(bounded_levenshtein(query, catalog_item_3, threshold=3)) # Выведет 4

2
4


**Асимптотическая сложность:**

* **Временная сложность:** $O(k \cdot \min(N, M))$, где $N$ и $M$ — длины строк, а $k$ — пороговое значение (`threshold`). В классическом Левенштейне сложность составляет $O(N \cdot M)$. Благодаря коридору вычислений мы на каждой из $\approx M$ итераций проходим только по окну шириной $2k + 1$. Раннее отсечение дополнительно ускоряет работу для совершенно непохожих строк до $O(1)$ (отсечение по длине) или $O(k^2)$ (отсечение на первых строках).
* **Пространственная сложность:** $O(\min(N, M))$, так как мы храним в памяти не всю матрицу, а только две последние строки (предыдущую и текущую).

####  Комментарий AI-тренера (Почему этот ответ идеален):
1. **Глубокое понимание алгоритмики:** Модель не просто написала вложенные циклы, а применила `Banded Levenshtein` (алгоритм Укконена). Большинство junior-разработчиков (и базовых LLM) справляются с отсечением только по длине строк, забывая про сужение окна поиска.
2. **Безопасность и чистота кода:** Присутствуют Type Hints (тайп-хинты), подробный docstring и защита от переполнения памяти (перестановка строк `s1` и `s2` для минимизации массива).
3. **Точность математических формулировок:** Асимптотика расписана строго и корректно. Оценка времени $O(k \cdot \min(N, M))$ обоснована математически, а не просто констатирована как факт.

